# Taller 2 — Notebook 02: Limpieza de los datos

**MINE-4101 · Predicción de churn para FinAlpes** · **Integrantes:** Daniel Triviño · Nicolás Bedoya

Aplica las decisiones del `01` y guarda `data/processed/churn_limpio.parquet`:

1. `country` y `gender` a su variante más común.
2. Eliminar duplicados exactos.
3. Eliminar filas fuera de rango o con `tenure > age`.
4. `credit_score` y `age` a entero.
5. Los faltantes se conservan: se imputan en el pipeline del modelo, solo con entrenamiento, para evitar fuga de
   información.

> Ejecutar después de `01_entendimiento`.

In [1]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import calidad as cal

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_theme(style='whitegrid')

RUTA_DATOS = '../churn.csv'
RUTA_SALIDA = '../data/processed/churn_limpio.parquet'
RUTA_FIGS = '../figs'
os.makedirs(os.path.dirname(RUTA_SALIDA), exist_ok=True)
os.makedirs(RUTA_FIGS, exist_ok=True)


## 1. Reglas y carga

Las mismas reglas del `01` (secciones 5.3 y 5.4): si cambian allá, hay que cambiarlas aquí.

In [2]:
# Abreviaturas que son el mismo valor
EQUIVALENCIAS = {'gender': {'M': 'Male', 'F': 'Female'}}

# Rangos válidos: (mínimo, máximo); None = sin límite. balance no lleva rango (un saldo puede ser negativo).
RANGOS = {
    'credit_score': (150, 950),     # escala de DataCrédito Experian
    'age': (18, 100),               # mayoría de edad y tope de plausibilidad
    'tenure': (0, None),            # la antigüedad no puede ser negativa
    'estimated_salary': (0, None),  # un salario no puede ser negativo
}

# Valores permitidos en las categóricas
DOMINIOS = {
    'country': ['France', 'Germany', 'Spain'],
    'gender': ['Female', 'Male'],
}

# Reglas entre columnas: True en las filas que las INCUMPLEN
REGLAS_CRUZADAS = {
    'antigüedad mayor que la edad (tenure > age)': lambda d: d['tenure'] > d['age'],
}


In [3]:
df = pd.read_csv(RUTA_DATOS)
n_inicial = len(df)
churn_inicial = df['churn'].mean()
print(f'Filas iniciales: {n_inicial:,}  |  columnas: {df.shape[1]}  |  tasa de churn: {churn_inicial * 100:.2f}%')

# Bitácora: cada paso deja registro de cuántas filas había, cuántas quedaron y cuántos valores cambió
bitacora = []
def registrar(paso, filas_antes, filas_despues, valores_modificados=0):
    bitacora.append({'paso': paso, 'filas_antes': filas_antes, 'filas_despues': filas_despues,
                     'filas_eliminadas': filas_antes - filas_despues, 'valores_modificados': valores_modificados})


Filas iniciales: 10,045  |  columnas: 11  |  tasa de churn: 20.36%


## 2. Paso 1 — Formato de `country` y `gender`

In [4]:
_, variantes = cal.consistencia_formato(df, columnas=['country', 'gender'], equivalencias=EQUIVALENCIAS)

# Para cada grupo, la variante más común es el valor final
principal = variantes[variantes['es_la_mas_comun']].set_index(['columna', 'grupo'])['variante']
variantes['pasa_a'] = [principal[(c, g)] for c, g in zip(variantes['columna'], variantes['grupo'])]
display(variantes)

antes = df[['country', 'gender']].copy()
for col in ['country', 'gender']:
    mapa = variantes[variantes['columna'] == col].set_index('variante')['pasa_a'].to_dict()
    df[col] = df[col].replace(mapa)
n_cambiados = int((antes != df[['country', 'gender']]).sum().sum())

print(f'Valores corregidos: {n_cambiados}')
for col in ['country', 'gender']:
    print(f'{col}:', df[col].value_counts().to_dict())
registrar('1. Formato de country y gender a la variante más común', len(df), len(df), n_cambiados)


,columna,grupo,variante,n,es_la_mas_comun,pasa_a
0,country,france,France,5003,True,France
1,country,france,FRANCE,22,False,France
2,country,france,france,17,False,France
3,country,germany,Germany,2502,True,Germany
4,country,germany,GERMANY,9,False,Germany
5,country,germany,germany,6,False,Germany
6,country,spain,Spain,2475,True,Spain
7,country,spain,SPAIN,9,False,Spain
8,country,spain,spain,2,False,Spain
9,gender,female,Female,4551,True,Female


Valores corregidos: 95
country: {'France': 5042, 'Germany': 2517, 'Spain': 2486}
gender: {'Male': 5480, 'Female': 4565}


## 3. Paso 2 — Duplicados exactos

Se conserva la primera aparición.

In [5]:
hall_dup, _ = cal.unicidad(df)
display(hall_dup)

n_antes = len(df)
df = df.drop_duplicates(keep='first')
print(f'Copias eliminadas: {n_antes - len(df)}')
registrar('2. Duplicados exactos (se conserva la primera aparición)', n_antes, len(df))


,dimension,columna,chequeo,n_afectados,pct_afectados,ejemplos
0,unicidad,todas las columnas,filas duplicadas (exactas),45,0.45,


Copias eliminadas: 45


## 4. Paso 3 — Filas con valores sin sentido

Los atípicos plausibles del IQR se conservan.

In [6]:
_, fuera_rango = cal.validez_rangos(df, RANGOS, devolver_filas=True)
idx_rango = set().union(*[set(f.index) for f in fuera_rango.values()])

idx_regla = set()
for regla in REGLAS_CRUZADAS.values():
    idx_regla |= set(df.index[regla(df).fillna(False).astype(bool)])

idx_sin_sentido = sorted(idx_rango | idx_regla)
print(f'Fuera de rango: {len(idx_rango)}  |  incumplen la regla: {len(idx_regla)}  |  '
      f'de estas, no estaban ya fuera de rango: {len(idx_regla - idx_rango)}')
print(f'Filas a eliminar: {len(idx_sin_sentido)}')
with pd.option_context('display.max_rows', None):
    display(df.loc[idx_sin_sentido])

n_antes = len(df)
df = df.drop(index=idx_sin_sentido)
registrar('3. Valores sin sentido (fuera de rango o tenure > age)', n_antes, len(df))


Fuera de rango: 14  |  incumplen la regla: 4  |  de estas, no estaban ya fuera de rango: 1
Filas a eliminar: 15


,credit_score,country,gender,age,tenure,balance,products_number,credit_card,active_member,estimated_salary,churn
992,850.00,France,Male,130.00,5,0.00,2,1,1,"44,827.47",0
1776,702.00,France,Female,5.00,6,"149,218.39",1,1,1,"9,633.01",0
1795,"1,200.00",Spain,Female,39.00,9,0.00,2,1,0,"28,368.37",0
1819,516.00,France,Female,0.00,9,0.00,2,0,1,"33,266.29",0
1881,120.00,Germany,Female,38.00,8,"103,653.51",2,1,1,"137,079.86",0
1984,587.00,Germany,Male,46.00,9,"107,850.82",1,1,0,"-5,000.00",1
2348,773.00,France,Female,50.00,4,0.00,2,1,0,-120.50,0
4031,433.00,France,Female,210.00,10,0.00,1,1,1,"87,711.61",0
4534,635.00,Spain,Male,150.00,2,0.00,2,0,0,"117,173.80",0
4679,811.00,Germany,Female,28.00,4,"167,738.82",2,1,1,-1.00,0


## 5. Paso 4 — Tipos

In [7]:
for col in ['credit_score', 'age']:
    assert (df[col].dropna() % 1 == 0).all(), f'{col} tiene valores que no son enteros'
    df[col] = df[col].astype('Int64')

df = df.reset_index(drop=True)
registrar('4. credit_score y age a entero (admite faltantes)', len(df), len(df))
df.dtypes.to_frame('tipo')


,tipo
credit_score,Int64
country,object
gender,object
age,Int64
tenure,int64
balance,float64
products_number,int64
credit_card,int64
active_member,int64
estimated_salary,float64


## 6. Paso 5 — Faltantes (se conservan)

In [8]:
falt = cal.completitud(df)
display(falt[falt['n_afectados'] > 0])
con_faltante = df.isna().any(axis=1)
print(f'Filas con algún faltante: {con_faltante.sum():,} ({con_faltante.mean() * 100:.1f}%)')


,dimension,columna,chequeo,n_afectados,pct_afectados,ejemplos
0,completitud,credit_score,valores faltantes,250,2.50,
3,completitud,age,valores faltantes,300,3.00,
5,completitud,balance,valores faltantes,200,2.00,
9,completitud,estimated_salary,valores faltantes,350,3.51,


Filas con algún faltante: 1,100 (11.0%)


## 7. Verificación

Solo deben quedar los faltantes; si aparece otra cosa, el `assert` detiene el notebook.

In [9]:
verificacion = cal.catalogo(
    cal.completitud(df),
    cal.unicidad(df)[0],
    cal.unicidad(df, normalizar_texto=True)[0],
    cal.validez_rangos(df, RANGOS),
    cal.validez_dominio(df, DOMINIOS, EQUIVALENCIAS),
    cal.consistencia_formato(df, equivalencias=EQUIVALENCIAS)[0],
    cal.consistencia_reglas(df, REGLAS_CRUZADAS),
)
display(verificacion)

pendientes = verificacion[verificacion['dimension'] != 'completitud']
assert pendientes.empty, f'Quedaron problemas sin resolver:\n{pendientes}'
print('Verificación OK: solo quedan los faltantes, que se imputan en el pipeline del modelo.')


,dimension,columna,chequeo,n_afectados,pct_afectados,ejemplos
0,completitud,estimated_salary,valores faltantes,350,3.51,
1,completitud,age,valores faltantes,300,3.00,
2,completitud,credit_score,valores faltantes,250,2.50,
3,completitud,balance,valores faltantes,200,2.00,


Verificación OK: solo quedan los faltantes, que se imputan en el pipeline del modelo.


## 8. Resumen y exportación

In [10]:
display(pd.DataFrame(bitacora))

n_final = len(df)
print(f'Filas: {n_inicial:,} -> {n_final:,}  (se eliminaron {n_inicial - n_final}, '
      f'{(n_inicial - n_final) / n_inicial * 100:.2f}%)')
print(f'Tasa de churn: {churn_inicial * 100:.2f}% -> {df["churn"].mean() * 100:.2f}%')


,paso,filas_antes,filas_despues,filas_eliminadas,valores_modificados
0,1. Formato de country y gender a la variante m...,10045,10045,0,95
1,2. Duplicados exactos (se conserva la primera ...,10045,10000,45,0
2,3. Valores sin sentido (fuera de rango o tenur...,10000,9985,15,0
3,4. credit_score y age a entero (admite faltantes),9985,9985,0,0


Filas: 10,045 -> 9,985  (se eliminaron 60, 0.60%)
Tasa de churn: 20.36% -> 20.38%


In [11]:
df.to_parquet(RUTA_SALIDA, index=False)
print(f'Guardado en {RUTA_SALIDA}  |  {df.shape[0]:,} filas x {df.shape[1]} columnas')


Guardado en ../data/processed/churn_limpio.parquet  |  9,985 filas x 11 columnas
